# Sonora Crop-Yield Estimator (SIAP 2021–2025)

**Goal**: Estimate pre-harvest agricultural crop yield (`Rendimiento`, tonnes per hectare) in **Sonora, Mexico** from municipality, crop, production cycle, irrigation/rainfed modality, and planned planted area (`Sembrada`).

> **Informational Use**: Pre-harvest planning estimate based on historical DGSIAP/SIAP Sonora records; not a financial, insurance, or production guarantee.

> **Evaluation Discipline**: Models are trained on `2021–2023` and selected strictly on `2024` validation RMSE. `train.py` is the canonical single `2025` held-out test evaluation and `2021–2024` refit path; this notebook does not tune or adapt models on `2025`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

manifest = json.loads(Path('data_manifest.json').read_text(encoding='utf-8'))
df = pd.read_csv('data/sonora_crop_yield_2021_2025.csv')
print('Derived dataset shape:', df.shape)
print('Filtering audit counts:', manifest['audit_counts'])
print('Rows per year:', df['Anio'].value_counts().sort_index().to_dict())
df.head()

Derived dataset shape: (5635, 7)
Filtering audit counts: {'sonora_total': 5782, 'dropped_non_tonelada': 4, 'dropped_nan_rendimiento': 143}
Rows per year: {2021: 952, 2022: 1190, 2023: 1321, 2024: 1127, 2025: 1045}


,Anio,Nommunicipio,Nomcicloproductivo,Nommodalidad,Nomcultivo,Sembrada,Rendimiento
0,2021,Puerto Peñasco,Perennes,Riego,Espárrago,440.0,10.77
1,2021,General Plutarco Elías Calles,Otoño-Invierno,Riego,Avena forrajera en verde,18.0,18.00
2,2021,General Plutarco Elías Calles,Otoño-Invierno,Riego,Trigo grano,226.0,5.10
3,2021,General Plutarco Elías Calles,Primavera-Verano,Riego,Algodón hueso,90.0,3.90
4,2021,General Plutarco Elías Calles,Primavera-Verano,Riego,Sorgo forrajero en verde,15.0,22.67


## 1. Exploratory Data Analysis (EDA)
Check missingness, cardinalities, numeric ranges (`Sembrada`, `Rendimiento`), the `Cosechada == 0` audit from `data_manifest.json`, and valid high-yield outliers.

In [2]:
print('Missing values:\n', df.isna().sum())
cat_cols = ['Nommunicipio', 'Nomcicloproductivo', 'Nommodalidad', 'Nomcultivo']
print('\nCategorical cardinalities:\n', df[cat_cols].nunique())
print('\nNumeric summary:\n', df[['Sembrada', 'Rendimiento']].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))
print('\nTop 5 high-yield observations (valid irrigated Pepino):')
df.sort_values('Rendimiento', ascending=False).head(5)

Missing values:
 Anio                  0
Nommunicipio          0
Nomcicloproductivo    0
Nommodalidad          0
Nomcultivo            0
Sembrada              0
Rendimiento           0
dtype: int64

Categorical cardinalities:
 Nommunicipio          71
Nomcicloproductivo     3
Nommodalidad           2
Nomcultivo            97
dtype: int64

Numeric summary:
            Sembrada  Rendimiento
count   5635.000000  5635.000000
mean     426.590760    19.974025
std     2090.008499    19.402191
min        0.330000     0.000000
5%         3.000000     1.330000
25%       13.000000     7.200000
50%       50.000000    16.400000
75%      175.500000    26.000000
95%     1230.300000    47.311000
max    42000.000000   198.000000

Top 5 high-yield observations (valid irrigated Pepino):


,Anio,Nommunicipio,Nomcicloproductivo,Nommodalidad,Nomcultivo,Sembrada,Rendimiento
757,2021,San Ignacio Río Muerto,Otoño-Invierno,Riego,Pepino,5.0,198.00
3192,2023,San Ignacio Río Muerto,Otoño-Invierno,Riego,Pepino,2.0,193.00
1843,2022,San Ignacio Río Muerto,Otoño-Invierno,Riego,Pepino,4.0,190.00
4245,2024,Etchojoa,Otoño-Invierno,Riego,Pepino,8.0,176.25
3572,2024,Imuris,Otoño-Invierno,Riego,Pepino,8.0,167.50


### Audit Notes
- **Zero-harvested (`Cosechada == 0`) rows**: `prepare_data.py` audited **143** `Sonora` + `Tonelada` rows where `Rendimiento` was `NaN`; 100% had `Cosechada == 0` (total crop loss → undefined `Volumen / 0`) and were excluded.
- **High-yield outliers (`167.5–198.0` t/ha)**: All top rows are irrigated `Pepino` (cucumber), which is biologically valid in Sonora protected/irrigated agriculture. We retain them and report both **RMSE** (primary, sensitive to large errors) and **MAE** (supporting, median-scale error).

## 2. Temporal Split, Baselines, and Validation Selection (`2021–2023` Train / `2024` Val)

In [3]:
num_cols = ['Sembrada']
features = num_cols + cat_cols
target = 'Rendimiento'

train_df = df[df['Anio'] <= 2023].copy()
val_df = df[df['Anio'] == 2024].copy()
print(f'Train (2021-2023): {len(train_df)} rows | Val (2024): {len(val_df)} rows')

def make_prep():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ])

def score_val(y_true, y_pred):
    p = np.clip(y_pred, 0.0, None)
    return {'rmse': round(float(root_mean_squared_error(y_true, p)), 4), 'mae': round(float(mean_absolute_error(y_true, p)), 4)}

g_mean = train_df[target].mean()
crop_means = train_df.groupby('Nomcultivo')[target].mean().to_dict()
results = [
    {'model': 'GlobalMean', **score_val(val_df[target], np.full(len(val_df), g_mean))},
    {'model': 'PerCropMean', **score_val(val_df[target], val_df['Nomcultivo'].map(crop_means).fillna(g_mean).to_numpy())},
]

lin = Pipeline([('prep', make_prep()), ('reg', LinearRegression())]).fit(train_df[features], train_df[target])
results.append({'model': 'LinearRegression', **score_val(val_df[target], lin.predict(val_df[features]))})

best_rf = None
for depth in [16, None]:
    for msl in [2, 5]:
        rf = Pipeline([('prep', make_prep()), ('reg', RandomForestRegressor(n_estimators=200, max_depth=depth, min_samples_leaf=msl, random_state=42, n_jobs=-1))])
        rf.fit(train_df[features], train_df[target])
        m = score_val(val_df[target], rf.predict(val_df[features]))
        results.append({'model': f'RandomForest(depth={depth},msl={msl})', **m})
        if best_rf is None or m['rmse'] < best_rf[0]:
            best_rf = (m['rmse'], f'RandomForest(depth={depth},msl={msl})', rf)

for lr in [0.05, 0.1]:
    for md in [3, 5]:
        gb = Pipeline([('prep', make_prep()), ('reg', GradientBoostingRegressor(n_estimators=200, learning_rate=lr, max_depth=md, random_state=42))])
        gb.fit(train_df[features], train_df[target])
        m = score_val(val_df[target], gb.predict(val_df[features]))
        results.append({'model': f'GradientBoosting(lr={lr},md={md})', **m})

pd.DataFrame(results).sort_values('rmse')

Train (2021-2023): 3463 rows | Val (2024): 1127 rows


,model,rmse,mae
5,"RandomForest(depth=None,msl=2)",8.4841,3.6285
10,"GradientBoosting(lr=0.1,md=5)",8.4971,4.4041
8,"GradientBoosting(lr=0.05,md=5)",9.4462,5.4504
9,"GradientBoosting(lr=0.1,md=3)",9.4664,5.6036
3,"RandomForest(depth=16,msl=2)",9.6439,5.5102
6,"RandomForest(depth=None,msl=5)",9.8220,5.1123
2,LinearRegression,9.9028,5.6620
4,"RandomForest(depth=16,msl=5)",10.3460,6.2560
7,"GradientBoosting(lr=0.05,md=3)",10.6811,6.7442
1,PerCropMean,10.9703,6.1305


## 3. Feature Importance Inspection (2024 Validation)
Permutation importance on the `2024` validation set confirms which input columns drive yield predictions.

In [4]:
_, best_label, best_pipe = best_rf
perm = permutation_importance(
    best_pipe, val_df[features], val_df[target],
    n_repeats=10, random_state=42, scoring='neg_root_mean_squared_error'
)
imp_df = pd.DataFrame({
    'feature': features,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)
print('Validation-selected model:', best_label)
imp_df

Validation-selected model: RandomForest(depth=None,msl=2)


,feature,importance_mean,importance_std
4,Nomcultivo,16.419138,0.589715
1,Nommunicipio,2.971476,0.275248
2,Nomcicloproductivo,1.159089,0.147770
3,Nommodalidad,0.969203,0.127818
0,Sembrada,0.629435,0.263524
